# Phase 4: Phân tích mạng tương tác protein-protein

Notebook này đọc protein ứng viên từ Phase 3, lấy các cạnh STRING liên quan, lọc cạnh tin cậy bằng `edge_weight_protein >= 0.4`, rồi tính đặc trưng mạng cho từng protein ứng viên.

Nguyên tắc chạy:
- Chỉ đọc dữ liệu từ `analysis` và `refined`.
- Không sửa raw/refined.
- Không dùng join điều kiện `OR` trên bảng cạnh lớn; tách cạnh theo src/dst rồi union.
- Output ghi vào `hdfs://master11:9000/drugtarget/data/analysis/protein_network_features`.

In [1]:
from pyspark.sql import SparkSession, functions as F

# Khai báo đường dẫn HDFS cho Phase 4.
HDFS_BASE = "hdfs://master11:9000"
DEG_PROTEINS_PATH = f"{HDFS_BASE}/drugtarget/data/analysis/deg_mapped_proteins"
EDGES_PROTEIN_PATH = f"{HDFS_BASE}/drugtarget/data/refined/STRING/edges_protein"
NODES_PROTEIN_PATH = f"{HDFS_BASE}/drugtarget/data/refined/STRING/nodes_protein"
NETWORK_OUTPUT_PATH = f"{HDFS_BASE}/drugtarget/data/analysis/protein_network_features"

# Ngưỡng dùng theo thang edge_weight_protein 0-1.
EDGE_WEIGHT_THRESHOLD = 0.4
HIGH_CONFIDENCE_THRESHOLD = 0.7

# Tạo SparkSession để xử lý mạng protein-protein.
spark = (
    SparkSession.builder.appName("drugtarget-gdc-phase4-ppi-network-analysis")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .enableHiveSupport()
    .getOrCreate()
)

# Giảm log để output notebook tập trung vào kết quả chính.
spark.sparkContext.setLogLevel("WARN")

print(f"Spark master: {spark.sparkContext.master}")
print(f"Input DEG protein: {DEG_PROTEINS_PATH}")
print(f"Input STRING edges: {EDGES_PROTEIN_PATH}")
print(f"Input STRING nodes: {NODES_PROTEIN_PATH}")
print(f"Output Phase 4: {NETWORK_OUTPUT_PATH}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/06/03 08:16:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark master: local[*]
Input DEG protein: hdfs://master11:9000/drugtarget/data/analysis/deg_mapped_proteins
Input STRING edges: hdfs://master11:9000/drugtarget/data/refined/STRING/edges_protein
Input STRING nodes: hdfs://master11:9000/drugtarget/data/refined/STRING/nodes_protein
Output Phase 4: hdfs://master11:9000/drugtarget/data/analysis/protein_network_features


In [2]:
def require_columns(frame, required_columns, frame_name: str) -> None:
    """Dừng pipeline nếu DataFrame thiếu cột bắt buộc."""
    missing_columns = sorted(set(required_columns) - set(frame.columns))
    if missing_columns:
        missing_text = ", ".join(missing_columns)
        raise ValueError(f"{frame_name} thiếu cột bắt buộc: {missing_text}")


# Đọc input Phase 4 từ HDFS.
deg_proteins = spark.read.parquet(DEG_PROTEINS_PATH)
edges = spark.read.parquet(EDGES_PROTEIN_PATH)
nodes = spark.read.parquet(NODES_PROTEIN_PATH)

DEG_REQUIRED_COLUMNS = ["protein_id"]
EDGES_REQUIRED_COLUMNS = [
    "protein_id_src",
    "protein_id_dst",
    "combined_score_protein",
    "edge_weight_protein",
]
NODES_REQUIRED_COLUMNS = [
    "protein_id",
    "ensp_id",
    "protein_name",
    "gene_name",
    "degree_protein",
    "weighted_degree_protein",
]
require_columns(deg_proteins, DEG_REQUIRED_COLUMNS, "deg_mapped_proteins")
require_columns(edges, EDGES_REQUIRED_COLUMNS, "STRING.edges_protein")
require_columns(nodes, NODES_REQUIRED_COLUMNS, "STRING.nodes_protein")

print("Schema deg_mapped_proteins:")
deg_proteins.printSchema()
print("Schema edges_protein:")
edges.printSchema()
print("Schema nodes_protein:")
nodes.printSchema()

Schema deg_mapped_proteins:
root
 |-- gene_id_base: string (nullable = true)
 |-- gene_name: string (nullable = true)
 |-- protein_id: string (nullable = true)
 |-- ensp_id: string (nullable = true)
 |-- gene_confidence: string (nullable = true)
 |-- log2FC: double (nullable = true)
 |-- p_value: double (nullable = true)
 |-- deg_direction: string (nullable = true)

Schema edges_protein:
root
 |-- protein_id_src: string (nullable = true)
 |-- protein_id_dst: string (nullable = true)
 |-- ensp_id_src: string (nullable = true)
 |-- ensp_id_dst: string (nullable = true)
 |-- combined_score_protein: double (nullable = true)
 |-- edge_weight_protein: double (nullable = true)

Schema nodes_protein:
root
 |-- protein_id: string (nullable = true)
 |-- ensp_id: string (nullable = true)
 |-- protein_name: string (nullable = true)
 |-- gene_id: string (nullable = true)
 |-- gene_name: string (nullable = true)
 |-- gene_name_norm: string (nullable = true)
 |-- degree_protein: long (nullable = true

In [3]:
# Lấy danh sách protein ứng viên không trùng lặp.
candidate_proteins = (
    deg_proteins.select("protein_id")
    .filter(F.col("protein_id").isNotNull())
    .distinct()
    .cache()
)

num_candidates = candidate_proteins.count()
if num_candidates == 0:
    raise ValueError("Không có protein ứng viên để phân tích mạng STRING.")
print(f"Số protein ứng viên Phase 4: {num_candidates}")

# Lọc cạnh liên quan protein ứng viên bằng hai nhánh src/dst để tránh join OR trên bảng lớn.
candidate_src = F.broadcast(candidate_proteins.withColumnRenamed("protein_id", "candidate_protein_id"))
candidate_dst = F.broadcast(candidate_proteins.withColumnRenamed("protein_id", "candidate_protein_id"))

edges_related_src = (
    edges.join(candidate_src, edges["protein_id_src"] == F.col("candidate_protein_id"), "inner")
    .drop("candidate_protein_id")
)
edges_related_dst = (
    edges.join(candidate_dst, edges["protein_id_dst"] == F.col("candidate_protein_id"), "inner")
    .drop("candidate_protein_id")
)

edges_related = edges_related_src.unionByName(edges_related_dst).distinct()

# Lọc cạnh có trọng số đủ tin cậy theo thang 0-1.
edges_filtered = edges_related.filter(F.col("edge_weight_protein") >= F.lit(EDGE_WEIGHT_THRESHOLD)).cache()

num_related_edges = edges_related.count()
num_filtered_edges = edges_filtered.count()
print(f"Số cạnh liên quan candidate trước lọc: {num_related_edges}")
print(f"Số cạnh sau lọc edge_weight >= {EDGE_WEIGHT_THRESHOLD}: {num_filtered_edges}")

# Chuyển cạnh vô hướng thành dạng dài để tính feature theo từng protein.
edges_as_src = edges_filtered.select(
    F.col("protein_id_src").alias("protein_id"),
    F.col("protein_id_dst").alias("neighbor_protein_id"),
    "combined_score_protein",
    "edge_weight_protein",
)
edges_as_dst = edges_filtered.select(
    F.col("protein_id_dst").alias("protein_id"),
    F.col("protein_id_src").alias("neighbor_protein_id"),
    "combined_score_protein",
    "edge_weight_protein",
)

protein_interactions_long = edges_as_src.unionByName(edges_as_dst).distinct()

# Chỉ giữ tương tác mà protein trung tâm là candidate.
candidate_interactions = protein_interactions_long.join(
    F.broadcast(candidate_proteins), on="protein_id", how="inner"
)

# Tính feature mạng cho từng protein ứng viên.
interaction_features = (
    candidate_interactions.groupBy("protein_id")
    .agg(
        F.countDistinct("neighbor_protein_id").cast("long").alias("num_interactions_in_deg_network"),
        F.avg("combined_score_protein").alias("avg_combined_score"),
        F.max("combined_score_protein").alias("max_combined_score"),
        F.countDistinct(
            F.when(F.col("edge_weight_protein") >= F.lit(HIGH_CONFIDENCE_THRESHOLD), F.col("neighbor_protein_id"))
        ).cast("long").alias("num_high_confidence_edges"),
    )
)

OUTPUT_COLUMNS = [
    "protein_id",
    "ensp_id",
    "protein_name",
    "gene_name",
    "degree_protein",
    "weighted_degree_protein",
    "num_interactions_in_deg_network",
    "avg_combined_score",
    "max_combined_score",
    "num_high_confidence_edges",
]

# Join với nodes_protein và thay đặc trưng mạng null bằng 0.
protein_network_features = (
    candidate_proteins.join(nodes, on="protein_id", how="left")
    .join(interaction_features, on="protein_id", how="left")
    .select(
        "protein_id",
        "ensp_id",
        "protein_name",
        "gene_name",
        F.coalesce(F.col("degree_protein"), F.lit(0)).cast("long").alias("degree_protein"),
        F.coalesce(F.col("weighted_degree_protein"), F.lit(0.0)).cast("double").alias("weighted_degree_protein"),
        F.coalesce(F.col("num_interactions_in_deg_network"), F.lit(0)).cast("long").alias("num_interactions_in_deg_network"),
        F.coalesce(F.col("avg_combined_score"), F.lit(0.0)).cast("double").alias("avg_combined_score"),
        F.coalesce(F.col("max_combined_score"), F.lit(0.0)).cast("double").alias("max_combined_score"),
        F.coalesce(F.col("num_high_confidence_edges"), F.lit(0)).cast("long").alias("num_high_confidence_edges"),
    )
    .select(*OUTPUT_COLUMNS)
    .cache()
)

output_count = protein_network_features.count()
if output_count != num_candidates:
    raise AssertionError(f"Số protein output Phase 4 không khớp candidate: output={output_count}, candidate={num_candidates}")

print("Schema output Phase 4:")
protein_network_features.printSchema()
print(f"Số dòng output Phase 4: {output_count}")

Số protein ứng viên Phase 4: 2579


26/06/03 08:16:53 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/03 08:16:53 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


Số cạnh liên quan candidate trước lọc: 3686380
Số cạnh sau lọc edge_weight >= 0.4: 484444


Schema output Phase 4:
root
 |-- protein_id: string (nullable = true)
 |-- ensp_id: string (nullable = true)
 |-- protein_name: string (nullable = true)
 |-- gene_name: string (nullable = true)
 |-- degree_protein: long (nullable = false)
 |-- weighted_degree_protein: double (nullable = false)
 |-- num_interactions_in_deg_network: long (nullable = false)
 |-- avg_combined_score: double (nullable = false)
 |-- max_combined_score: double (nullable = false)
 |-- num_high_confidence_edges: long (nullable = false)

Số dòng output Phase 4: 2579


In [4]:
# Ghi kết quả đặc trưng mạng vào layer analysis.
protein_network_features.write.mode("overwrite").option("compression", "snappy").parquet(NETWORK_OUTPUT_PATH)
print(f"Đã ghi output Phase 4: {NETWORK_OUTPUT_PATH}")

Đã ghi output Phase 4: hdfs://master11:9000/drugtarget/data/analysis/protein_network_features


In [5]:
# Kiểm tra output path tồn tại trên HDFS.
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
output_hdfs_path = spark._jvm.org.apache.hadoop.fs.Path(NETWORK_OUTPUT_PATH)
output_fs = output_hdfs_path.getFileSystem(hadoop_conf)
if not output_fs.exists(output_hdfs_path):
    raise AssertionError(f"Output Phase 4 chưa tồn tại trên HDFS: {NETWORK_OUTPUT_PATH}")

# Đọc lại output và kiểm tra schema/logic cơ bản.
network_output = spark.read.parquet(NETWORK_OUTPUT_PATH)
if network_output.columns != OUTPUT_COLUMNS:
    raise AssertionError(f"Schema Phase 4 không đúng: {network_output.columns}")

bad_high_confidence_count = network_output.filter(
    F.col("num_high_confidence_edges") > F.col("num_interactions_in_deg_network")
).count()
if bad_high_confidence_count != 0:
    raise AssertionError(f"Có {bad_high_confidence_count} dòng high confidence lớn hơn tổng interaction.")

null_numeric_count = network_output.filter(
    F.col("degree_protein").isNull()
    | F.col("weighted_degree_protein").isNull()
    | F.col("num_interactions_in_deg_network").isNull()
    | F.col("avg_combined_score").isNull()
    | F.col("max_combined_score").isNull()
    | F.col("num_high_confidence_edges").isNull()
).count()
if null_numeric_count != 0:
    raise AssertionError(f"Có {null_numeric_count} dòng feature số bị null.")

print("Kiểm tra output Phase 4 hoàn tất.")
print("Top protein theo weighted_degree_protein:")
network_output.orderBy(F.desc("weighted_degree_protein")).show(20, truncate=False)
print("Top protein theo num_interactions_in_deg_network:")
network_output.orderBy(F.desc("num_interactions_in_deg_network")).show(20, truncate=False)
print("Số protein không có interaction sau lọc:")
print(network_output.filter(F.col("num_interactions_in_deg_network") == 0).count())

Kiểm tra output Phase 4 hoàn tất.
Top protein theo weighted_degree_protein:


+--------------------+---------------+------------+---------+--------------+-----------------------+-------------------------------+------------------+------------------+-------------------------+
|protein_id          |ensp_id        |protein_name|gene_name|degree_protein|weighted_degree_protein|num_interactions_in_deg_network|avg_combined_score|max_combined_score|num_high_confidence_edges|
+--------------------+---------------+------------+---------+--------------+-----------------------+-------------------------------+------------------+------------------+-------------------------+
|9606.ENSP00000380070|ENSP00000380070|NULL        |GAPDH    |9357          |5700.620000000002      |1937                           |550.8156943727414 |999.0             |278                      |
|9606.ENSP00000360266|ENSP00000360266|NULL        |JUN      |5439          |3249.7599999999984     |1088                           |600.4034926470588 |999.0             |255                      |
|9606.ENSP00000

+--------------------+---------------+------------+---------+--------------+-----------------------+-------------------------------+------------------+------------------+-------------------------+
|protein_id          |ensp_id        |protein_name|gene_name|degree_protein|weighted_degree_protein|num_interactions_in_deg_network|avg_combined_score|max_combined_score|num_high_confidence_edges|
+--------------------+---------------+------------+---------+--------------+-----------------------+-------------------------------+------------------+------------------+-------------------------+
|9606.ENSP00000380070|ENSP00000380070|NULL        |GAPDH    |9357          |5700.620000000002      |1937                           |550.8156943727414 |999.0             |278                      |
|9606.ENSP00000385675|ENSP00000385675|NULL        |IL6      |4611          |3189.512               |1377                           |604.3848946986202 |999.0             |376                      |
|9606.ENSP00000

2


In [6]:
# Giải phóng cache sau khi hoàn tất Phase 4.
edges_filtered.unpersist()
protein_network_features.unpersist()
candidate_proteins.unpersist()
print("Hoàn tất Phase 4: Phân tích mạng tương tác protein-protein")

Hoàn tất Phase 4: Phân tích mạng tương tác protein-protein
